# 03 - Caching: Worked Example

This notebook walks through the two-level embedding cache step by step.

**Before running:** start the stack from the module folder:

```bash
docker compose up --build
```

The gateway listens on `localhost:8081`.

In [ ]:
import json
import time

import httpx

BASE = "http://localhost:8081"
client = httpx.Client(timeout=30)

# Verify the stack is healthy
resp = client.get(f"{BASE}/health")
print(resp.json())

## 1. First request — cache miss

The cache is empty. The gateway calls the embedding API (500 ms simulated latency) and stores the result.

In [ ]:
def embed(query: str) -> dict:
    resp = client.post(f"{BASE}/v1/embeddings", json={"query": query})
    resp.raise_for_status()
    data = resp.json()
    print(f"cache={data['cache']:<4}  type={str(data.get('cache_type')):<8}  latency={data['latency_ms']} ms")
    return data

result = embed("how to reset my password")

Notice `cache=miss` and latency around 500 ms — the full API round-trip.

## 2. Exact cache hit — same query

Send the identical string. The gateway finds the SHA-256 key in Redis and returns immediately.

In [ ]:
result = embed("how to reset my password")
# Expect: cache=hit, cache_type=exact, latency ~1 ms

Under 5 ms. No API call was made.

## 3. Exact hit is case- and whitespace-insensitive

The gateway normalises the query before hashing: `strip().lower()`. Different capitalisation hits the same key.

In [ ]:
result = embed("How To Reset My Password")
# Still an exact hit — same normalised hash

## 4. Semantic cache hit — different wording, same intent

A paraphrase shares no exact key with the stored entry. The gateway:

1. Calls the embedding API to get a vector for this new query.
2. Scans the semantic index and computes cosine similarity.
3. Finds a stored embedding above the 0.97 threshold and returns it.
4. Writes an exact-match entry so the next identical request is O(1).

In [ ]:
result = embed("steps to reset password")
print(f"  matched_query : {result.get('matched_query')}")
print(f"  similarity    : {result.get('similarity')}")

In [ ]:
# Now the promoted exact key exists — next identical request is instant
result = embed("steps to reset password")
# Expect: cache=hit, cache_type=exact

## 5. Cache miss — a novel, unrelated query

A billing question lives in a completely different semantic cluster. Cosine similarity with the password entry will be low — a genuine miss.

In [ ]:
result = embed("how do I cancel my subscription")
# Expect: cache=miss, new entry stored

## 6. Build up a realistic traffic pattern

Simulate 10 requests mixing repeated and novel queries and observe the hit rate.

In [ ]:
# First, clear state so this cell is reproducible
client.delete(f"{BASE}/cache")

queries = [
    "how to reset my password",        # 1 — miss
    "how to reset my password",        # 2 — exact hit
    "steps to reset password",         # 3 — semantic hit
    "forgot my password",              # 4 — semantic hit (same cluster)
    "how do I cancel my subscription", # 5 — miss
    "cancel my subscription",          # 6 — semantic hit
    "billing inquiry about invoice",   # 7 — miss
    "how to reset my password",        # 8 — exact hit
    "invoice for last month",          # 9 — semantic hit
    "login credentials not working",   # 10 — semantic hit (auth cluster)
]

print(f"{'#':<3} {'cache':<6} {'type':<10} {'latency ms':>10}  query")
print("-" * 70)
for i, q in enumerate(queries, 1):
    r = client.post(f"{BASE}/v1/embeddings", json={"query": q}).json()
    print(f"{i:<3} {r['cache']:<6} {str(r.get('cache_type')):<10} {r['latency_ms']:>10}  {q}")

## 7. Cache statistics

In [ ]:
stats = client.get(f"{BASE}/cache/stats").json()
print(json.dumps(stats, indent=2))

print(f"\nHit rate: {stats['hit_rate']:.0%}")
print(f"Exact hits:    {stats['exact_hits']}")
print(f"Semantic hits: {stats['semantic_hits']}")
print(f"Misses:        {stats['misses']}")

## 8. Inspect Redis directly

Run these in your terminal to see what Redis is holding:

```bash
# List all cache keys
docker compose exec redis redis-cli KEYS 'cache:*'

# Show all semantic entry IDs
docker compose exec redis redis-cli SMEMBERS cache:semantic:index

# Inspect one semantic entry (replace <id> with an ID from the set above)
docker compose exec redis redis-cli HGETALL cache:embedding:semantic:<id>

# Check TTL on an exact entry
docker compose exec redis redis-cli TTL cache:embedding:exact:<sha256>
```

Here we do it programmatically via the gateway stats endpoint:

In [ ]:
# Show the embedding vectors for comparison
import math

def cosine_sim(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    mag_a = math.sqrt(sum(x * x for x in a))
    mag_b = math.sqrt(sum(y * y for y in b))
    return dot / (mag_a * mag_b) if mag_a and mag_b else 0.0

password_emb = client.post(f"{BASE}/v1/embeddings", json={"query": "how to reset my password"}).json()["embedding"]
billing_emb  = client.post(f"{BASE}/v1/embeddings", json={"query": "how do I cancel my subscription"}).json()["embedding"]
forgot_emb   = client.post(f"{BASE}/v1/embeddings", json={"query": "forgot my password"}).json()["embedding"]

print(f"password  vs  cancel subscription : {cosine_sim(password_emb, billing_emb):.4f}  (different clusters)")
print(f"password  vs  forgot my password  : {cosine_sim(password_emb, forgot_emb):.4f}  (same cluster)")

## 9. Latency comparison

Measure the p50 and p99 latency for hits versus misses over multiple requests.

In [ ]:
import statistics

# Clear and seed the cache with one entry
client.delete(f"{BASE}/cache")
client.post(f"{BASE}/v1/embeddings", json={"query": "how to reset my password"})

hit_latencies  = []
miss_latencies = []

for _ in range(20):
    r = client.post(f"{BASE}/v1/embeddings", json={"query": "how to reset my password"}).json()
    hit_latencies.append(r["latency_ms"])

for i in range(5):
    r = client.post(f"{BASE}/v1/embeddings", json={"query": f"unique query number {i} about widgets"}).json()
    miss_latencies.append(r["latency_ms"])

def percentile(data, p):
    s = sorted(data)
    idx = max(0, int(len(s) * p / 100) - 1)
    return s[idx]

print(f"Cache HIT  — p50: {percentile(hit_latencies, 50):.1f} ms   p99: {percentile(hit_latencies, 99):.1f} ms")
print(f"Cache MISS — p50: {percentile(miss_latencies, 50):.1f} ms   p99: {percentile(miss_latencies, 99):.1f} ms")
print(f"\nSpeedup: {percentile(miss_latencies, 50) / max(percentile(hit_latencies, 50), 0.1):.0f}x faster on cache hit")

## 10. Reset the environment

In [ ]:
resp = client.delete(f"{BASE}/cache")
print(resp.json())

stats = client.get(f"{BASE}/cache/stats").json()
print(stats)  # All counters and entries should be zero

## Key observations

| Scenario | Latency | API calls |
| --- | ---: | ---: |
| First request for a query | ~510 ms | 1 |
| Exact repeat | ~1 ms | 0 |
| Paraphrase (semantic hit) | ~510 ms first time, ~1 ms after promotion | 1 then 0 |
| Unrelated novel query | ~510 ms | 1 |

On a typical support chatbot where 10 % of all queries are unique:
- Without cache: 1,000 requests → 1,000 API calls
- With cache: 1,000 requests → ~100 API calls (90 % saved)

Semantic caching extends coverage to paraphrases, reducing the unique-query fraction further.